In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split

import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [11]:
train_df = pd.read_csv('/content/drive/MyDrive/seoul-landmark-teamproject/dataset/dataset/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/seoul-landmark-teamproject/dataset/dataset/test.csv')

train_df['file_path'] = '/content/drive/MyDrive/seoul-landmark-teamproject/dataset/dataset/train' + train_df['file_name']
test_df['file_path'] = '/content/drive/MyDrive/seoul-landmark-teamproject/dataset/dataset/test' + test_df['file_name']

print(train_df.head())
print(test_df.head())

  file_name  label                                          file_path
0   001.PNG      9  /content/drive/MyDrive/seoul-landmark-teamproj...
1   002.PNG      4  /content/drive/MyDrive/seoul-landmark-teamproj...
2   003.PNG      1  /content/drive/MyDrive/seoul-landmark-teamproj...
3   004.PNG      1  /content/drive/MyDrive/seoul-landmark-teamproj...
4   005.PNG      6  /content/drive/MyDrive/seoul-landmark-teamproj...
  file_name                                          file_path
0   001.PNG  /content/drive/MyDrive/seoul-landmark-teamproj...
1   002.PNG  /content/drive/MyDrive/seoul-landmark-teamproj...
2   003.PNG  /content/drive/MyDrive/seoul-landmark-teamproj...
3   004.PNG  /content/drive/MyDrive/seoul-landmark-teamproj...
4   005.PNG  /content/drive/MyDrive/seoul-landmark-teamproj...


In [12]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

print(f'학습 데이터: {len(train_data)}장')
print(f'검증 데이터: {len(val_data)}장')

학습 데이터: 578장
검증 데이터: 145장


In [13]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['file_path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if 'label' in self.df.columns:
            label = self.df.iloc[idx]['label']
            return image, label

        return image

train_dataset = CustomDataset(train_data.reset_index(drop=True), transform=train_transform)
val_dataset = CustomDataset(val_data.reset_index(drop=True), transform=val_transform)
test_dataset = CustomDataset(test_df.reset_index(drop=True), transform=val_transform)

print(f'train dataset: {len(train_dataset)}장')
print(f'val dataset: {len(val_dataset)}장')
print(f'test dataset: {len(test_dataset)}장')

train dataset: 578장
val dataset: 145장
test dataset: 199장


In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(f'train_loader: {len(train_loader)}배치')
print(f'val_loader: {len(val_loader)}배치')
print(f'test_loader: {len(test_loader)}배치')

train_loader: 19배치
val_loader: 5배치
test_loader: 7배치


In [15]:
# 첫 번째 배치 꺼내서 확인
images, labels = next(iter(train_loader))
print(f"이미지 shape: {images.shape}")  # (32, 3, 224, 224)
print(f"라벨: {labels}")

# 첫 번째 배치에서 이미지 가져오기
images, labels = next(iter(train_loader))

# Normalize 되돌리기 (안 하면 색이 이상하게 보임)
def denormalize(tensor):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# 이미지 6장 출력
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i].clone())
    img = img.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(f'label: {labels[i].item()}')
    ax.axis('off')

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/seoul-landmark-teamproject/dataset/dataset/train343.PNG'

In [ ]:
### 전이학습 ###
# ResNet18 사전학습 모델 불러오기
model_fe = models.resnet18(pretrained=True)   # Feature Extraction용
model_ft = models.resnet18(pretrained=True)   # Fine-tuning용

# 마지막 레이어 교체 (1000 → 10 클래스)
model_fe.fc = nn.Linear(512, 10) # Feature Extraction 용 /backbone 전체 freeze /마지막 레이어(fc)만 학습
model_ft.fc = nn.Linear(512, 10) # Fine-tuning 용 /전체 레이어 다 학습

# backbone 전체 freeze
for param in model_fe.parameters():
    param.requires_grad = False

# 마지막 레이어만 학습 가능하게
for param in model_fe.fc.parameters():
    param.requires_grad = True

def train_model(model, train_loader, val_loader, epochs=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

    for epoch in range(epochs):
        # 학습
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # 검증
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {100*correct/total:.2f}%")

    return model

In [ ]:
# 전처리 (ResNet은 224x224 필요)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("=== Feature Extraction ===")
model_fe = train_model(model_fe, train_loader, val_loader, epochs=5)

print("\n=== Fine-tuning ===")
model_ft = train_model(model_ft, train_loader, val_loader, epochs=5)